# MMPose vs MediaPipe Pose Comparison

Run this notebook on Google Colab with a T4 GPU runtime. It writes MMPose artifacts next to the existing MediaPipe pose artifacts and produces rule/classifier comparison tables.

### 1. Mount Drive

In [37]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 2. Install MMPose

In [38]:
# Colab Python 3.12-friendly RTMW/RTMPose runtime. Avoid mmcv/mmdet native ops.
# Install rtmlib first, then force the GPU ONNX Runtime wheel last. Both CPU/GPU wheels expose `onnxruntime`.
# Restart the runtime after this cell if onnxruntime was already imported.
!pip uninstall -y -q openmim openxlab chumpy mmpose mmdet mmcv mmcv-lite mmengine xtcocotools onnxruntime onnxruntime-gpu
!pip install -q -U pip wheel "setuptools>=69"
!pip install -q -U rtmlib opencv-python numpy tqdm
!pip uninstall -y -q onnxruntime onnxruntime-gpu
!pip install -q --no-cache-dir --force-reinstall "onnxruntime-gpu[cuda,cudnn]"
!python -m pip show onnxruntime onnxruntime-gpu || true

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rtmlib 0.0.15 requires onnxruntime, which is not installed.
google-cloud-aiplatform 1.148.1 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.35.0 which is incompatible.
google-cloud-bigtable 2.36.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.35.0 which is incompatible.
torch 2.10.0+cu128 requires nvidia-cublas-cu12==12.8.4.1; platform_system == "Linux", but you have nvidia-cublas-cu12 12.9.2.10 which is incompatible.
torch 2.10.0+cu128 requires nvidia-cuda-nvrtc-cu12==12.8.93; platform_system == "Linux", but you have nvidia-cuda-nvrtc-cu12 12.9.86 which is incompatible.
torch 2.10.0+cu128 requires nvidia-cuda-runtime-cu12==12.8.90; platform_system == "Linux", but you have nvidia-cuda-r

In [39]:
import torch
import onnxruntime as ort
import rtmlib
from rtmlib import Wholebody

if hasattr(ort, "preload_dlls"):
    ort.preload_dlls()

providers = ort.get_available_providers()
print("cuda_available:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("torch:", torch.__version__)
print("onnxruntime:", ort.__version__)
print("onnxruntime providers:", providers)
print("rtmlib:", getattr(rtmlib, "__version__", "unknown"))

if not torch.cuda.is_available():
    raise RuntimeError("Colab GPU is not available. Change Runtime > Change runtime type > GPU.")
if "CUDAExecutionProvider" not in providers:
    raise RuntimeError("ONNX Runtime CUDAExecutionProvider is missing. Restart runtime, rerun install cell, then rerun this check before extraction.")

print("rtmlib Wholebody import: OK")

cuda_available: True
gpu: Tesla T4
torch: 2.10.0+cu128
onnxruntime: 1.26.0
onnxruntime providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
rtmlib: unknown
rtmlib Wholebody import: OK


### 3. Configure Paths

In [40]:
from pathlib import Path
import subprocess

PROJECT_ROOT = Path('/content/drive/MyDrive/x-coach')
DATASET_ROOT = PROJECT_ROOT / 'data' / 'Fitness-AQA_dataset_release' / 'Squat' / 'Labeled_Dataset'
OUTPUT_ROOT = PROJECT_ROOT / 'data' / 'Squat' / 'Labeled_Dataset'
LABEL_DIR = DATASET_ROOT / 'Labels'
SPLIT_DIR = DATASET_ROOT / 'Splits'
LOCAL_VIDEO_DIR = Path('/content/local_videos')
LOCAL_VIDEO_DIR.mkdir(parents=True, exist_ok=True)

video_zip = DATASET_ROOT / 'videos.zip'
if video_zip.exists():
    subprocess.run(['unzip', '-q', '-n', str(video_zip), '-d', str(LOCAL_VIDEO_DIR)], check=True)

VIDEO_ROOT = LOCAL_VIDEO_DIR if any(LOCAL_VIDEO_DIR.rglob('*.mp4')) else DATASET_ROOT / 'videos'
MMPOSE_JSON_DIR = OUTPUT_ROOT / 'mmpose_pose_json'
MMPOSE_FEATURE_DIR = OUTPUT_ROOT / 'mmpose_pose_features'
MMPOSE_VIEW_METADATA = OUTPUT_ROOT / 'mmpose_view_metadata.csv'
MMPOSE_RULE_DIR = OUTPUT_ROOT / 'mmpose_pose_rule_detections'
MMPOSE_RULE_SUMMARY = OUTPUT_ROOT / 'mmpose_pose_rule_detections_summary.csv'
MMPOSE_RULE_METRICS = OUTPUT_ROOT / 'mmpose_pose_rule_validation_metrics.csv'
MMPOSE_CLASSIFIER_ROOT = PROJECT_ROOT / 'data' / 'Squat' / 'mmpose_pose_classifier_experiments'

def first_existing(*paths):
    for path in paths:
        if path.exists():
            return path
    return paths[0]

FORWARD_LABELS = first_existing(LABEL_DIR / 'error_forward.json', LABEL_DIR / 'error_knees_forward.json')
INWARD_LABELS = first_existing(LABEL_DIR / 'error_inward.json', LABEL_DIR / 'error_knees_inward.json')

print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATASET_ROOT:', DATASET_ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('VIDEO_ROOT:', VIDEO_ROOT)
print('FORWARD_LABELS:', FORWARD_LABELS)
print('INWARD_LABELS:', INWARD_LABELS)

PROJECT_ROOT: /content/drive/MyDrive/x-coach
DATASET_ROOT: /content/drive/MyDrive/x-coach/data/Fitness-AQA_dataset_release/Squat/Labeled_Dataset
OUTPUT_ROOT: /content/drive/MyDrive/x-coach/data/Squat/Labeled_Dataset
VIDEO_ROOT: /content/local_videos
FORWARD_LABELS: /content/drive/MyDrive/x-coach/data/Fitness-AQA_dataset_release/Squat/Labeled_Dataset/Labels/error_knees_forward.json
INWARD_LABELS: /content/drive/MyDrive/x-coach/data/Fitness-AQA_dataset_release/Squat/Labeled_Dataset/Labels/error_knees_inward.json


### 4. Extract MMPose Whole-Body JSON

In [ ]:
cmd = [
    'python', 'src/run_mmpose_pose_extraction.py',
    '--video-dir', str(VIDEO_ROOT),
    '--split-dir', str(SPLIT_DIR),
    '--output-dir', str(MMPOSE_JSON_DIR),
    '--model', 'wholebody',
    '--device', 'cuda:0',
]
subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

# !python "/content/drive/MyDrive/x-coach/src/run_mmpose_pose_extraction.py" \
#   --video-dir "/content/local_videos" \
#   --split-dir "/content/drive/MyDrive/x-coach/data/Fitness-AQA_dataset_release/Squat/Labeled_Dataset/Splits" \
#   --output-dir "/content/drive/MyDrive/x-coach/data/Squat/Labeled_Dataset/mmpose_pose_json" \
#   --model wholebody \
#   --device cuda:0

### 5. Convert MMPose JSON to Pose Features

In [ ]:
cmd = [
    'python', 'scripts/run_pose_feature_extraction.py',
    '--pose-json-dir', str(MMPOSE_JSON_DIR),
    '--split-dir', str(SPLIT_DIR),
    '--output-dir', str(MMPOSE_FEATURE_DIR),
    '--overwrite',
]
subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

### 6. View Metadata and Rule Evaluation

In [46]:
subprocess.run([
    'python', 'src/run_view_estimation.py',
    '--pose-json-dir', str(MMPOSE_JSON_DIR),
    '--split-dir', str(SPLIT_DIR),
    '--output', str(MMPOSE_VIEW_METADATA),
], cwd=PROJECT_ROOT, check=True)

subprocess.run([
    'python', 'src/run_pose_rule_detection.py',
    '--pose-json-dir', str(MMPOSE_JSON_DIR),
    '--split-dir', str(SPLIT_DIR),
    '--output-dir', str(MMPOSE_RULE_DIR),
    '--summary-output', str(MMPOSE_RULE_SUMMARY),
    '--no-retrieval',
], cwd=PROJECT_ROOT, check=True)

!python /content/drive/MyDrive/x-coach/src/evaluate_pose_rule_detection.py \
  --detections-dir {MMPOSE_RULE_DIR} \
  --view-metadata {MMPOSE_VIEW_METADATA} \
  --forward-labels {FORWARD_LABELS} \
  --inward-labels {INWARD_LABELS} \
  --output {MMPOSE_RULE_METRICS}

Saved validation metrics to /content/drive/MyDrive/x-coach/data/Squat/Labeled_Dataset/mmpose_pose_rule_validation_metrics.csv
| class_id | view_type | n | precision | recall | f1 | mean_segment_iou | tp | fp | tn | fn |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| knees_forward | ALL | 1623 | 1.000000 | 0.004509 | 0.008977 | 0.000904 | 5 | 0 | 514 | 1104 |
| knees_forward | rear | 824 | 0.000000 | 0.000000 | 0.000000 | 0.000000 | 0 | 0 | 260 | 564 |
| knees_forward | rear_oblique | 627 | 0.000000 | 0.000000 | 0.000000 | 0.000000 | 0 | 0 | 204 | 423 |
| knees_forward | side | 170 | 1.000000 | 0.041322 | 0.079365 | 0.008285 | 5 | 0 | 49 | 116 |
| knees_forward | unknown | 2 | 0.000000 | 0.000000 | 0.000000 | 0.000000 | 0 | 0 | 1 | 1 |
| knees_inward | ALL | 1623 | 0.226121 | 0.500000 | 0.311409 | 0.054972 | 116 | 397 | 994 | 116 |
| knees_inward | rear | 824 | 0.307692 | 0.478632 | 0.374582 | 0.078412 | 56 | 126 | 581 | 61 |
| knees_inward | rear_oblique | 627 | 

### 7. Train MMPose Pose-Only Classifiers

In [ ]:
subprocess.run([
    'python', 'scripts/run_videomae_experiment_grid.py',
    '--feature-dir', str(MMPOSE_FEATURE_DIR),
    '--train-keys', str(SPLIT_DIR / 'train_keys.json'),
    '--val-keys', str(SPLIT_DIR / 'val_keys.json'),
    '--test-keys', str(SPLIT_DIR / 'test_keys.json'),
    '--forward-labels', str(FORWARD_LABELS),
    '--inward-labels', str(INWARD_LABELS),
    '--output-root', str(MMPOSE_CLASSIFIER_ROOT),
    '--label-modes', 'combined,knees_forward,knees_inward',
    '--seeds', '1,2,3,4,5',
    '--epochs', '20',
    '--lr', '3e-4',
    '--hidden-dim', '128',
    '--dropout', '0.4',
    '--weight-decay', '0.01',
    '--early-stopping-patience', '5',
    '--threshold-objective', 'balanced_accuracy',
    '--device', 'cuda',
    '--normalize-features',
], cwd=PROJECT_ROOT, check=True)

### 8. Write Backend Comparison Report

In [ ]:
subprocess.run([
    'python', 'scripts/compare_pose_backends.py',
    '--mmpose-pose-json-dir', str(MMPOSE_JSON_DIR),
    '--mmpose-rule-metrics', str(MMPOSE_RULE_METRICS),
    '--mmpose-classifier-summary', str(MMPOSE_CLASSIFIER_ROOT / 'metrics' / 'experiment_summary.csv'),
], cwd=PROJECT_ROOT, check=True)

comparison_md = PROJECT_ROOT / 'data' / 'Squat' / 'mmpose_mediapipe_comparison' / 'backend_comparison.md'
print(comparison_md.read_text()[:4000])